In [ ]:

import os
import cv2
import numpy as np
import random
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model


In [ ]:

TRAIN_DIR = "archive/dataset/train"
VAL_DIR = "archive/dataset/val"

IMG_SIZE = 128
SCALE = 4

EPOCHS = 5
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5)


In [ ]:

def load_paths(directory):
    paths = []
    for root, _, files in os.walk(directory):
        for f in files:
            if f.lower().endswith((".png",".jpg",".jpeg")):
                paths.append(os.path.join(root,f))
    return paths

train_paths = load_paths(TRAIN_DIR)
val_paths = load_paths(VAL_DIR)

batch_size = 4 if len(train_paths) >= 4 else len(train_paths)


In [ ]:

def norm(x):
    return x/127.5 - 1

def make_pair(p):
    img = cv2.imread(p)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    hr = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
    lr = cv2.resize(hr,(IMG_SIZE//SCALE,IMG_SIZE//SCALE))
    lr = cv2.resize(lr,(IMG_SIZE,IMG_SIZE))

    return norm(lr), norm(hr)


In [ ]:

def data_gen(paths, bs):
    while True:
        batch = random.choices(paths,k=min(bs,len(paths)))
        x,y = [],[]
        for p in batch:
            a,b = make_pair(p)
            x.append(a)
            y.append(b)
        yield np.array(x), np.array(y)


In [ ]:

def show(lr,sr,hr):
    lr = (lr+1)/2
    sr = (sr+1)/2
    hr = (hr+1)/2

    plt.figure(figsize=(12,4))
    for i,img in enumerate([lr,sr,hr]):
        plt.subplot(1,3,i+1)
        plt.imshow(img)
        plt.axis("off")
    plt.show()


In [ ]:

def build_model():
    inp = layers.Input(shape=(IMG_SIZE,IMG_SIZE,3))
    x = layers.Conv2D(64,9,padding="same",activation="relu")(inp)
    return Model(inp,x)

model = build_model()
model.summary()
